# **Spritpreisprognose Machine Learning**

### Abschlussprojekt DSI Data Science Institute by Fabian Rappert  
Weiterbildung 6 Monate Data Science

### Projektziel
Der hypothetische Use-Case für das Projekt ist die Frage eines Autofahrers oder einer Autofahrerin bei annähernd leerem Tank (Reichweite 100 km), mit welcher Tankstrategie sich die Kosten minimieren lassen (buy the dip). Da sich die Preise dynamisch und auf den ersten Blick möglicherweise intransparent ändern, ist das nicht offensichtlich. Mit diesem Tool soll eine Indikation gegeben werden, wann im Prognosezeitraum der wahrscheinlich beste Zeitpunkt wäre.

### Projektrahmen
Zunächst wird nur die ARAL Tankstelle an der Dürener Straße 407 in Köln betrachtet. Ghislain wohnt in der Nähe und kann explorativ die Daten der Anzeigetafel mit den gelieferten Daten vergleichen, um Hinweise auf die Datenqualität zu erhalten.

### Datenquellen

| Quelle | Inhalt | Frequenz | Zugang |
|--------|--------|----------|--------|
| [Tankerkönig Open Data](https://creativecommons.tankerkoenig.de/) | Kraftstoffpreise deutscher Tankstellen (ab 2014) | täglich | kostenlos, API-Key nötig |
| [Yahoo Finance (BZ=F)](https://finance.yahoo.com/quote/BZ=F) | Brent Crude Oil Futures – Tagespreise (ab 2014) | täglich | kostenlos, kein Key |
| [Yahoo Finance (BZ=F)](https://finance.yahoo.com/quote/BZ=F) | Brent Crude Oil Futures – Intraday | stündlich (letzte 60 Tage) | kostenlos, kein Key |
| [EZB Statistical Data Warehouse](https://data-api.ecb.europa.eu) | EUR/USD Referenzkurs | täglich | kostenlos, kein Key |
| [feiertage-api.de](https://feiertage-api.de/) | Gesetzliche Feiertage (alle 16 Bundesländer, ab 2014) | jährlich | kostenlos, kein Key |
| [OpenHolidays API](https://openholidaysapi.org/) | Schulferien (alle 16 Bundesländer, ab 2014) | jährlich | kostenlos, kein Key |
| Bundesministerium der Finanzen | Energiesteuer-Sätze (Änderungen ab 2014) | bei Änderung | manuell gepflegt |
| Umweltbundesamt | CO₂-Abgabe (BEHG-Preisstufen ab 2021) | jährlich | manuell gepflegt |
| diverse (ADAC, Presse, Destatis) | Externe Ereignisse (Krisen, Preisschocks) | ereignisbasiert | manuell gepflegt |

### Zeitraum für Machine-Learning
08.06.2014 - 31.12.2025 (Beginn der Daten von Tankerkönig.de bis Ende des Vorjahres) ggf. anpassen bei kürzerer Verfügbarkeit für betrachtete Tankstelle

Wichtige Ereignisse in dieser Zeit:
- **COVID-19 Lockdown** (16.03.–15.06.2020) — Mobilität kollabiert, Nachfrage unabhängig vom Ölpreis
- **Rhein-Niedrigwasser** (15.07.–15.10.2022) — Diesel +5–8 ct/L durch Logistikengpass NRW-spezifisch

# ML Projektstruktur – Spritpreisprognose

## 1. Datenquellen & Export (ETL-Output)

Granularität: eine Row = eine Preisänderung (minutengenau, Köln)

### Quelldateien
**Spritpreise**
- [ ] tankstellen_preise.parquet
- [ ] tankstellen_stationen.parquet

**Features**
- [ ] brent_futures_intraday_1h.csv
- [ ] brent_futures_daily.csv
- [ ] eur_usd_rate.csv
- [ ] energiesteuer.csv
- [ ] co2_abgabe.csv
- [ ] feiertage.csv
- [ ] schulferien.csv
- [ ] wetter_koeln.csv
- [ ] externe_effekte.csv

### Spaltenstruktur ML-Exportdatei (**optional für Post-MVP)
```
timestamp, station_uuid

── Zielvariablen ──────────────────────────────
e5
e10
diesel

── Features ───────
brent_futures_usd
eur_usd
is_feiertag
is_schulferien
temp_avg
niederschlag_mm
sonnenstunden
ist_tankrabatt
co2_preis_eur_t
is_lockdown
is_niedrigwasser
```

---

## 2. EDA für ML

- Verteilung der Zielvariablen (Histogramme, Boxplots)
- Zeitreihenplot: Trend, Saisonalität, Zyklen
- Autokorrelation (ACF/PACF) → Hinweise auf sinnvolle Lag-Längen
- Korrelationsmatrix: Features vs. Zielgröße
- Anomalien & Ausreißer identifizieren (Z-Score, IQR)
- Externe Schocks sichtbar machen (Corona, Ukraine-Krieg)
  → Entscheidung: rausfiltern oder als Feature behalten?

---

## 3. Feature Engineering & Selection

- Price-Lags: lag_1, lag_7, lag_14, lag_30
- Rolling Statistics: rolling_mean_7/30, rolling_std_7
- Zeitfeatures: hour, weekday, month, is_weekend
- Brent-Lags & Delta-Features
- EUR/USD-Lags
- is_feiertag_morgen, is_schulferien_morgen
- Feature Importance (nach erstem Modell-Training verfeinern)
- Multikollinearität prüfen (VIF)

---

## 4. ML Modellierung

**Ziel:** Δ Preis in 24h (preis_morgen - preis_jetzt) + Richtung (STEIGT / FÄLLT / GLEICH ±0.5ct)

- Train/Test-Split: zeitbasiert (kein random shuffle!)
  - Train 2014–2022 | Val 2023 | Test 2024
- Externe Schocks: als Dummy-Feature UND ausgeklammert testen
- Modellkandidaten:
  - Baseline: Naïve Forecast (gestern = heute)
  - XGBoost / LightGBM
  - AutoML: PyCaret oder AutoGluon
- Evaluation: MAE, RMSE, MAPE + Trefferquote Richtung
- Modellauswahl & Begründung dokumentieren
- Finales Modell als .pkl auf S3 speichern

---

## 5. Live-Dashboard (Streamlit + AWS S3)

**Kernfrage: Jetzt tanken oder morgen?**

### Tab 1: Empfehlung
- Große klare Aussage: „Jetzt tanken" / „Morgen tanken"
- Aktueller Preis vs. prognostizierter Preis in 24h
- Konfidenz / Unsicherheitsbereich
- Tankstelle auswählen (Köln, nach Name/Marke/PLZ)

### Tab 2: Prognosequalität
- delta_predicted vs. delta_actual (letzte 7/30 Tage)
- Trefferquote Richtung (%) als Hauptmetrik
- MAE absoluter Preis

---

## 6. Deployment

### Prediction Log
predictions.parquet im Repo (von GitHub Actions committed)

### GitHub Actions

forecast.yml   cron: '0 * * * *'   (stündlich)
               1. Aktuelle Features abrufen (Brent, EUR/USD, Wetter, ...)
               2. Modell (.pkl aus Repo) laden
               3. Prognose erstellen (Δ in 24h)
               4. Row in predictions.parquet schreiben
               5. Commit & Push ins Repo

actuals.yml    cron: '0 * * * *'   (stündlich)
               1. Offene Rows prüfen: target_timestamp <= now?
               2. Echten Preis via Tankerkönig Live-API abrufen
               3. delta_actual, direction_correct, MAE eintragen
               4. Commit & Push ins Repo

### Secrets
TANKERKOENIG_API_KEY


In [ ]:
#